# Stage B v5 — K=10000 with v3 tiering (vLLM optimized, no truncation)

**Hypothesis**: v3's tiered LLM judge (AUTO bypass + LLM-tier with evidence gate
+ DROP fail) achieved F1=0.0822 on the K=500 Stage A pool — where only 47% of
gold was reachable. Applying the same tiering to the K=10000 pool gives the
LLM access to ~84% of reachable gold instead. The bottleneck shifts from
"gold-not-in-pool" to "judge precision", which is what we want to test.

**What's new vs v3**:
- **K=10000** input (was K=2000), → ~100k candidates total across 10 val queries
- **Full 1200-char source text** in every prompt (v3 used 400 chars)
- **NO truncation anywhere**: `max_model_len=8192`, `max_tokens=512`, prompts
  never exceed model context, JSON responses never get cut off mid-token
- **`temperature=0.0`** for deterministic verdicts (v3 used 0.7 which adds noise
  to a binary-classification task)
- **Chunked disk checkpointing** every 2000 LLM responses so an OOM or
  disconnect never loses more than ~5 min of progress
- **Resume-from-checkpoint** logic: re-running the inference cell picks up
  exactly where it left off

**Tiering** (mutually exclusive, applied in order):

| Tier | Rule | Action |
|---|---|---|
| **AUTO** | `article_match AND (co_cit ≥ 5 OR code_in_target)` | Keep, NO LLM call |
| **LLM** | passes evidence gate but not AUTO | LLM judges |
| **DROP** | fails evidence gate | Reject, NO LLM call |

Evidence gate = `article_match OR co_cit ≥ 1 OR concept_cosine ≥ 0.45 OR code_in_target`

**Final keep** = AUTO ∪ (LLM_keep ∧ quote_verbatim_in_text)

**Prerequisite**: `stage_b_input_k10k.parquet` must exist at
`/content/drive/MyDrive/swiss_law/research/stage_b_grounded_llm/stage_b_input_k10k.parquet`.
Run `precompute_stage_b_k10k_input.py` locally and upload that parquet first.

**Expected wall-clock**: ~1.5-2h on Blackwell (40-50k LLM-tier prompts × ~6/s).

## Phase 0 — Setup (single cell, validated upfront)

In [ ]:
# =============================================================================
# PHASE 0 — Setup (run from a FRESH KERNEL)
# =============================================================================

import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path

# CUDA allocator: reduce fragmentation OOM for big vLLM allocations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

IS_COLAB = "google.colab" in sys.modules
print(f"[setup] Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

def _pip(*args, check=True):
    return subprocess.run(
        [sys.executable, "-m", "pip", *args],
        check=check, capture_output=True, text=True,
    )

if IS_COLAB:
    print("[setup] removing torchcodec + bitsandbytes (preempt env breakage)")
    _pip("uninstall", "-y", "torchcodec",   check=False)
    _pip("uninstall", "-y", "bitsandbytes", check=False)

    print("[setup] installing vllm + ecosystem (~3-5 min) ...")
    t0 = time.time()
    r = _pip(
        "install", "-q", "-U",
        "vllm>=0.6.0",
        "pandas>=2.0", "pyarrow>=16.0", "tqdm",
        check=False,
    )
    print(f"  done in {(time.time()-t0)/60:.1f} min  (returncode={r.returncode})")
    if r.returncode != 0:
        print("=== stdout (tail) ==="); print(r.stdout[-1500:])
        print("=== stderr (tail) ==="); print(r.stderr[-1500:])
        raise RuntimeError("vllm install failed")

print("\n[setup] validating imports ...")

import torch
assert torch.cuda.is_available(), "FATAL: no CUDA"
print(f"  torch:    {torch.__version__}  CUDA: {torch.version.cuda}")
print(f"  GPU:      {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)")

import vllm
from vllm import LLM, SamplingParams
print(f"  vllm:     {vllm.__version__}")

import pandas as pd
import numpy as np
print(f"  pandas:   {pd.__version__}")
print(f"  numpy:    {np.__version__}")

print("\n[setup] OK")


## Phase 1 — Config + path verification

In [ ]:
# =============================================================================
# PHASE 1 — Config + path verification
# =============================================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/swiss_law")

# --- Inputs ----------------------------------------------------------------
STAGE_B_INPUT = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "stage_b_input_k10k.parquet"
VAL_CSV       = DRIVE_ROOT / "data" / "val.csv"
VAL_ASPECTS   = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"
# Gold sets — same path the reranker PoC used
GOLD_JSON     = DRIVE_ROOT / "v7_pool_recall_089_and_per_query" / "snapshot" / "gold_doc_sets.json"
# Fallback paths in case the user keeps gold elsewhere
GOLD_JSON_FALLBACKS = [
    DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json",
    DRIVE_ROOT / "research" / "v7_pool_recall_089_and_per_query" / "snapshot" / "gold_doc_sets.json",
]

# --- Outputs ---------------------------------------------------------------
OUT_DIR        = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "stage_b_v5_k10k_tiered_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUT_DIR / "llm_response_chunks"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_JSON   = OUT_DIR / "stage_b_v5_metrics.json"
PICKS_PARQ     = OUT_DIR / "stage_b_v5_picks.parquet"
TIER_JSON      = OUT_DIR / "stage_b_v5_tier_distribution.json"
RAW_LLM_PARQ   = OUT_DIR / "stage_b_v5_raw_llm_responses.parquet"

# --- vLLM config (no truncation anywhere) ---------------------------------
LLM_MODEL     = "Qwen/Qwen3-32B-AWQ"
MAX_MODEL_LEN = 8192    # prompts ~1k-2k tokens + 512 output = comfortable
MAX_TOKENS    = 512     # never truncate the JSON response
TEMPERATURE   = 0.0     # deterministic verdicts (v3 used 0.7; v5 uses 0)
GPU_MEM_UTIL  = 0.85

# --- Tier thresholds (matches v3 evidence gate) ---------------------------
AUTO_COCIT_THRESH       = 5      # AUTO needs cc >= 5 AND article_match
EVIDENCE_COCIT_THRESH   = 1      # LLM tier needs cc >= 1 (or article_match etc)
EVIDENCE_COSINE_THRESH  = 0.45   # LLM tier alternative entry

# --- Prompt config (full text, NO truncation in prompt) ------------------
MAX_QUESTION_CHARS = 1500
MAX_TEXT_CHARS_IN_PROMPT = 1200    # use FULL bundle text (v3 used 400)

# --- Inference chunking ---------------------------------------------------
LLM_CHUNK_SIZE = 2000    # disk-checkpoint every N prompts

# Resolve GOLD_JSON
if not GOLD_JSON.exists():
    for fb in GOLD_JSON_FALLBACKS:
        if fb.exists():
            GOLD_JSON = fb
            break

print(f"Inputs:")
for p, label in [(STAGE_B_INPUT, "stage_b_input_k10k.parquet"),
                 (VAL_CSV, "val.csv"),
                 (VAL_ASPECTS, "val_aspects.parquet"),
                 (GOLD_JSON, "gold_doc_sets.json")]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status:<7}] {label:<32} -> {p}")

if not STAGE_B_INPUT.exists():
    raise FileNotFoundError(
        f"stage_b_input_k10k.parquet not found at {STAGE_B_INPUT}.\n"
        f"Run `python precompute_stage_b_k10k_input.py` locally first, "
        f"then upload the parquet to:\n  {STAGE_B_INPUT.parent}/"
    )

print(f"\nOutputs (Drive):")
for p in [METRICS_JSON, PICKS_PARQ, TIER_JSON, RAW_LLM_PARQ]:
    print(f"  -> {p}")
print(f"  -> {CHECKPOINT_DIR}/chunk_NNNN.parquet  (intermediate)")

print(f"\nvLLM config: model={LLM_MODEL}")
print(f"  max_model_len={MAX_MODEL_LEN}, max_tokens={MAX_TOKENS}, "
      f"temp={TEMPERATURE}, gpu_mem={GPU_MEM_UTIL}")
print(f"\nTier thresholds: AUTO co_cit>={AUTO_COCIT_THRESH}, "
      f"evidence cosine>={EVIDENCE_COSINE_THRESH}")
print(f"Prompt: question<={MAX_QUESTION_CHARS} chars, text<={MAX_TEXT_CHARS_IN_PROMPT} chars")


## Phase 2 — Load K=10000 input + apply tiering

In [ ]:
# =============================================================================
# PHASE 2 — Load Stage B input + apply tiering (AUTO / LLM / DROP)
# =============================================================================

t0 = time.time()
df = pd.read_parquet(STAGE_B_INPUT)
print(f"Loaded {len(df):,} candidates from {STAGE_B_INPUT.name} in {time.time()-t0:.1f}s")

required = {"qid","did","citation","family","role","text",
            "article_match","co_citation_count","code_in_target",
            "concept_cosine_score","area_match","chamber_match",
            "is_BGE","best_aspect_id","is_gold","chamber","law_code"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"input parquet missing columns: {sorted(missing)}")

# Normalize bool columns to actual bool dtype (parquet can return numpy uint8)
for col in ["article_match","code_in_target","chamber_match","is_BGE",
            "area_match","is_gold","cc_hit_top","cc_hit_strong"]:
    if col in df.columns:
        df[col] = df[col].astype(bool)
df["co_citation_count"] = df["co_citation_count"].astype(int)
df["concept_cosine_score"] = df["concept_cosine_score"].astype(float)

print(f"\nPer-query input:")
print(df.groupby("qid").agg(
    rows=("did","count"),
    gold=("is_gold","sum"),
).to_string())

# --- Tiering (vectorized — fast on 100k rows) -------------------------------
# AUTO: article_match AND (co_cit >= 5 OR code_in_target)
is_auto = df["article_match"] & (
    (df["co_citation_count"] >= AUTO_COCIT_THRESH) | df["code_in_target"]
)

# Evidence gate: article_match OR co_cit >= 1 OR cosine >= 0.45 OR code_in_target
passes_gate = (
    df["article_match"]
    | (df["co_citation_count"] >= EVIDENCE_COCIT_THRESH)
    | (df["concept_cosine_score"] >= EVIDENCE_COSINE_THRESH)
    | df["code_in_target"]
)

df["tier"] = "DROP"
df.loc[passes_gate, "tier"] = "LLM"
df.loc[is_auto, "tier"] = "AUTO"

# Tier distribution
tier_table = df.groupby(["qid","tier"]).agg(
    rows=("did","count"), gold=("is_gold","sum"),
).reset_index()
print(f"\nTier distribution by query:")
print(tier_table.to_string(index=False))

gtots = df.groupby("tier").agg(rows=("did","count"), gold=("is_gold","sum")).reset_index()
print(f"\nGlobal tier counts:")
print(gtots.to_string(index=False))

# Persist tier distribution to disk
tier_dist = {
    "by_query": tier_table.to_dict(orient="records"),
    "totals":   gtots.to_dict(orient="records"),
    "thresholds": {
        "AUTO_COCIT_THRESH":      AUTO_COCIT_THRESH,
        "EVIDENCE_COCIT_THRESH":  EVIDENCE_COCIT_THRESH,
        "EVIDENCE_COSINE_THRESH": EVIDENCE_COSINE_THRESH,
    },
}
with open(TIER_JSON, "w") as f:
    json.dump(tier_dist, f, indent=2, default=str)
print(f"\n[checkpoint] tier distribution saved -> {TIER_JSON.name}")


## Phase 3 — Build prompts for LLM tier (v3 template + full text)

In [ ]:
# =============================================================================
# PHASE 3 — Build LLM prompts for LLM-tier candidates
# =============================================================================
# Verbatim v3 prompt template. Two changes vs v3:
#   - text excerpt is FULL 1200 chars (was 400 in v3)
#   - question NOT truncated below 1500 chars (val queries top out at ~1100)

val_df = pd.read_csv(VAL_CSV)
asp_df = pd.read_parquet(VAL_ASPECTS)
print(f"Loaded val.csv ({len(val_df)} rows), val_aspects ({len(asp_df)} rows)")

def aspects_for_qid(qid):
    row = asp_df[asp_df.query_id == qid]
    if row.empty:
        return "- a1 (w=1.00): (no decomposition available)"
    aspects = list(row.iloc[0]["aspects"])
    lines = []
    for a in aspects:
        aid = a.get("id", "?")
        lbl = a.get("label", "")
        w = float(a.get("weight", 0))
        lines.append(f"- {aid} (w={w:.2f}): {lbl}")
    return "\n".join(lines)

questions = {}
for _, r in val_df.iterrows():
    qid = r["query_id"]
    questions[qid] = {
        "text": str(r["query"])[:MAX_QUESTION_CHARS],
        "aspects": aspects_for_qid(qid),
    }
print(f"\nBuilt question+aspects for {len(questions)} queries")

# v3 prompt template (verbatim) with full 1200-char text excerpt
PROMPT_TEMPLATE = """You are a Swiss legal analyst. Decide whether the CANDIDATE below is a citation a competent Swiss lawyer would write in answering the LEGAL QUESTION.

LEGAL QUESTION:
{question}

QUERY ASPECTS (decompose the answer):
{aspects}

CANDIDATE:
- Citation: {citation}
- Family: {family}  {family_extra}
- Paragraph role: {role}
- Retrieval evidence:
  * Article-match with a query-named statute: {article_match}
  * Cited by {co_citation_count} other top-100 pool court paragraphs
  * Concept-cosine vs query aspects: {concept_cosine_score:.2f}  (best on aspect {best_aspect_id})
  * Code in query's legal area: {code_in_target}
  * Chamber matches legal area: {chamber_match}
- Text excerpt: "{text}"

DECISION RULES:
1. GOLD = a Swiss lawyer writing the legal answer would cite this.
2. Substantive paragraphs (legal_standard, reasoning, application, holding) are far more often gold than facts/costs/dispositif/procedural_history.
3. Court paragraphs from chambers irrelevant to the legal area are rarely gold.
4. Articles from codes outside the query's legal area are rarely gold (Art. 100 BGG is gold for any appeal question exception).
5. If retrieval evidence shows article_match=true OR co_citation_count >= 3 OR concept_cosine >= 0.60, bias toward YES.
6. If you say keep=true, you MUST quote a verbatim 5-30-word substring of the text excerpt that justifies inclusion.

OUTPUT STRICT JSON (no prose, begin with `{{`):
{{"keep": true|false, "confidence": <0..1>, "which_aspect": "<aspect id>", "evidence_quote": "<verbatim substring of text>"}}"""

def build_prompt(citation, family, chamber, law_code, role,
                 article_match, co_citation_count, concept_cosine_score,
                 best_aspect_id, code_in_target, chamber_match, text, q):
    if family == "court":
        family_extra = f"(chamber: {chamber or '?'})"
    else:
        family_extra = f"(code: {law_code or '?'})"
    return PROMPT_TEMPLATE.format(
        question=q["text"],
        aspects=q["aspects"],
        citation=citation,
        family=family,
        family_extra=family_extra,
        role=role or "?",
        article_match=str(bool(article_match)).lower(),
        co_citation_count=int(co_citation_count),
        concept_cosine_score=float(concept_cosine_score),
        best_aspect_id=best_aspect_id or "?",
        code_in_target=str(bool(code_in_target)).lower(),
        chamber_match=str(bool(chamber_match)).lower(),
        text=(text or "")[:MAX_TEXT_CHARS_IN_PROMPT].replace('"', "'"),
    )

# --- Build prompts for LLM tier only ---------------------------------------
llm_df = df[df.tier == "LLM"].reset_index(drop=True)
print(f"\nBuilding prompts for {len(llm_df):,} LLM-tier candidates ...")

t0 = time.time()
prompts = []
for r in llm_df.itertuples(index=False):
    q = questions[r.qid]
    prompts.append(build_prompt(
        r.citation, r.family, r.chamber, r.law_code,
        r.role, r.article_match, r.co_citation_count, r.concept_cosine_score,
        r.best_aspect_id, r.code_in_target, r.chamber_match, r.text, q,
    ))
print(f"  built {len(prompts):,} prompts in {time.time()-t0:.1f}s")

lens = [len(p) for p in prompts]
print(f"\nPrompt length stats (chars):")
print(f"  min:    {min(lens)}")
print(f"  median: {sorted(lens)[len(lens)//2]}")
print(f"  mean:   {sum(lens)//len(lens)}")
print(f"  max:    {max(lens)}")
print(f"  (assuming ~3 chars/token, max prompt ~= {max(lens)//3} tokens; "
      f"limit is {MAX_MODEL_LEN} - {MAX_TOKENS} = {MAX_MODEL_LEN-MAX_TOKENS})")

# --- Show one gold sample for sanity ---------------------------------------
gold_idx = next((i for i, r in enumerate(llm_df.itertuples()) if r.is_gold), None)
if gold_idx is not None:
    g_row = llm_df.iloc[gold_idx]
    print(f"\n=== Sample prompt for a GOLD candidate ({g_row['qid']}, {g_row['citation']}) ===")
    snippet = prompts[gold_idx]
    print(snippet[:2500])
    print("... [truncated for display]" if len(snippet) > 2500 else "")


## Phase 4 — vLLM batched inference (chunked + checkpointed)

In [ ]:
# =============================================================================
# PHASE 4 — vLLM batched inference, chunked + checkpointed
# =============================================================================
# This is the long step. Crashes/disconnects are safe: re-running picks up from
# the last completed chunk on disk (CHECKPOINT_DIR/chunk_NNNN.parquet).

print(f"Loading {LLM_MODEL} via vLLM ...")
print(f"  max_model_len={MAX_MODEL_LEN}, gpu_mem_util={GPU_MEM_UTIL}")
t0 = time.time()
llm = LLM(
    model=LLM_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    enforce_eager=False,
    trust_remote_code=True,
)
print(f"vLLM loaded in {time.time()-t0:.1f}s.  "
      f"GPU mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# SamplingParams: temperature=0 means top_p/top_k are ignored.
# stop tokens cut off after the JSON object closes, preventing trailing prose.
sp = SamplingParams(
    temperature=TEMPERATURE,
    top_p=1.0 if TEMPERATURE == 0.0 else 0.8,
    max_tokens=MAX_TOKENS,
    seed=42,
    stop=["</json>", "\n\n\n"],
)

# --- Resume from existing checkpoints --------------------------------------
existing_chunks = sorted(CHECKPOINT_DIR.glob("chunk_*.parquet"))
already_done = set()
for cp in existing_chunks:
    cdf = pd.read_parquet(cp)
    already_done.update(cdf.global_idx.tolist())
total = len(prompts)
todo_indices = [i for i in range(total) if i not in already_done]
print(f"\n[resume] checkpoints found: {len(existing_chunks)}, "
      f"prompts done: {len(already_done):,}, todo: {len(todo_indices):,}")

# --- Process in chunks ----------------------------------------------------
t_start = time.time()
chunk_counter = len(existing_chunks)
for chunk_start in range(0, len(todo_indices), LLM_CHUNK_SIZE):
    chunk_indices = todo_indices[chunk_start:chunk_start + LLM_CHUNK_SIZE]
    if not chunk_indices:
        break

    chunk_prompts = [prompts[i] for i in chunk_indices]
    chunk_t = time.time()

    messages = [[{"role": "user", "content": p}] for p in chunk_prompts]
    outs = llm.chat(
        messages,
        sampling_params=sp,
        chat_template_kwargs={"enable_thinking": False},
        use_tqdm=False,
    )
    raw_texts = [o.outputs[0].text if o.outputs else "" for o in outs]

    chunk_df = pd.DataFrame({
        "global_idx": chunk_indices,
        "raw_text":   raw_texts,
    })
    chunk_path = CHECKPOINT_DIR / f"chunk_{chunk_counter:04d}.parquet"
    chunk_df.to_parquet(chunk_path, index=False)
    chunk_counter += 1

    done = len(already_done) + chunk_start + len(chunk_indices)
    elapsed = (time.time() - t_start) / 60
    rate = (chunk_start + len(chunk_indices)) / max(0.01, time.time() - t_start)
    eta = (len(todo_indices) - chunk_start - len(chunk_indices)) / max(0.01, rate) / 60
    print(f"  chunk {chunk_counter-1:>4}: {len(chunk_indices)} prompts in "
          f"{time.time()-chunk_t:.1f}s  total={done}/{total}  "
          f"rate={rate:.1f}/s  elapsed={elapsed:.1f}min  eta={eta:.1f}min  "
          f"-> {chunk_path.name}")

print(f"\nLLM inference done in {(time.time()-t_start)/60:.1f} min "
      f"(total chunks: {chunk_counter})")

# --- Free vLLM before downstream parsing -----------------------------------
del llm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU mem after vLLM free: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


## Phase 5 — Parse responses + apply quote/evidence filters

In [ ]:
# =============================================================================
# PHASE 5 — Parse JSON + apply quote + evidence filters
# =============================================================================

# Load all chunks into one aligned DataFrame
chunk_paths = sorted(CHECKPOINT_DIR.glob("chunk_*.parquet"))
print(f"Loading {len(chunk_paths)} chunks ...")
chunk_dfs = [pd.read_parquet(cp) for cp in chunk_paths]
raw_df = pd.concat(chunk_dfs, ignore_index=True).sort_values("global_idx").reset_index(drop=True)
print(f"  total LLM responses: {len(raw_df):,}")
assert len(raw_df) == len(prompts), (
    f"chunk count mismatch: {len(raw_df)} vs {len(prompts)} prompts. "
    f"Re-run Phase 4 — some chunks may have failed."
)

def parse_json_lenient(s):
    """Lenient JSON parse — handles markdown fences, trailing commas, prose around."""
    if not s: return None
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*\n", "", s)
        s = re.sub(r"\n```\s*$", "", s)
    m = re.search(r"\{[\s\S]*?\}", s)
    if not m: return None
    body = m.group(0)
    body = re.sub(r",(\s*[}\]])", r"\1", body)
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        return None

print("Parsing JSON responses ...")
parses = [parse_json_lenient(t) for t in raw_df.raw_text.tolist()]
parse_ok = sum(1 for p in parses if p is not None)
print(f"  parse_ok: {parse_ok}/{len(parses)} = {parse_ok/len(parses):.4f}")

# --- Quote substring check (case-insensitive, whitespace-normalized) -------
def quote_in_text(quote, text):
    if not quote: return False
    qn = re.sub(r"\s+", " ", quote.strip().lower())
    tn = re.sub(r"\s+", " ", (text or "").lower())
    return bool(qn) and qn in tn

keep_llm, quote_ok, confidence, evidence_quote, which_aspect = [], [], [], [], []
for parsed, text in zip(parses, llm_df.text.tolist()):
    if parsed is None:
        keep_llm.append(False); quote_ok.append(False); confidence.append(0.0)
        evidence_quote.append(""); which_aspect.append("")
        continue
    k = bool(parsed.get("keep", False))
    q = str(parsed.get("evidence_quote", "")).strip()
    conf = float(parsed.get("confidence", 0.0) or 0.0)
    asp = str(parsed.get("which_aspect", "") or "")
    keep_llm.append(k)
    quote_ok.append(quote_in_text(q, text))
    confidence.append(conf)
    evidence_quote.append(q)
    which_aspect.append(asp)

llm_df = llm_df.copy()
llm_df["llm_keep"] = keep_llm
llm_df["quote_ok"] = quote_ok
llm_df["confidence"] = confidence
llm_df["evidence_quote"] = evidence_quote
llm_df["which_aspect"] = which_aspect
# LLM tier already passed the evidence gate via tier assignment.
# Final keep = LLM said yes AND its quote is verbatim in text.
llm_df["final_keep"] = llm_df["llm_keep"] & llm_df["quote_ok"]

# Persist raw + parsed responses for diagnosis
llm_df_with_raw = llm_df.copy()
llm_df_with_raw["raw_response"] = raw_df.raw_text.tolist()
llm_df_with_raw.to_parquet(RAW_LLM_PARQ, index=False)
print(f"\n[checkpoint] raw+parsed LLM responses -> {RAW_LLM_PARQ.name}")

# Diagnostics
n_llm   = len(llm_df)
n_keep  = int(llm_df.llm_keep.sum())
n_qok   = int(llm_df.quote_ok.sum())
n_final = int(llm_df.final_keep.sum())
print(f"\nLLM-tier diagnostics:")
print(f"  parse_ok rate:    {parse_ok/n_llm:.4f}")
print(f"  keep_llm rate:    {n_keep/n_llm:.4f}  ({n_keep}/{n_llm})")
print(f"  quote_in_text:    {n_qok/max(1,n_keep):.4f}  ({n_qok}/{n_keep} of LLM-yes)")
print(f"  final_keep rate:  {n_final/n_llm:.4f}  ({n_final}/{n_llm})")

# Per-query LLM-tier diagnostics
print(f"\nPer-query LLM-tier final keeps:")
print(llm_df.groupby("qid").agg(
    llm_tier_rows=("did","count"),
    llm_yes=("llm_keep","sum"),
    final_keep=("final_keep","sum"),
    gold_kept=("final_keep", lambda s: int((s & llm_df.loc[s.index,"is_gold"]).sum())),
).to_string())


## Phase 6 — Combine AUTO + LLM-keep into final picks

In [ ]:
# =============================================================================
# PHASE 6 — Combine AUTO + LLM-keep into final per-query picks
# =============================================================================

auto_picks = df[df.tier == "AUTO"][["qid","did","citation","is_gold"]].copy()
auto_picks["tier"]       = "AUTO"
auto_picks["llm_keep"]   = True
auto_picks["quote_ok"]   = True
auto_picks["confidence"] = 1.0
auto_picks["final_keep"] = True
auto_picks["evidence_quote"] = ""
auto_picks["which_aspect"]   = ""

llm_keep_cols = ["qid","did","citation","is_gold","tier",
                 "llm_keep","quote_ok","confidence","final_keep",
                 "evidence_quote","which_aspect"]
llm_picks = llm_df[llm_df.final_keep][llm_keep_cols].copy()

all_picks = pd.concat([auto_picks, llm_picks], ignore_index=True)
all_picks.to_parquet(PICKS_PARQ, index=False)

print(f"Final picks per query:")
summary = all_picks.groupby("qid").agg(
    picks=("did","count"),
    auto=("tier", lambda s: int((s=="AUTO").sum())),
    llm=("tier", lambda s: int((s=="LLM").sum())),
    gold_picks=("is_gold","sum"),
)
print(summary.to_string())
print(f"\n[checkpoint] picks -> {PICKS_PARQ.name}  ({len(all_picks):,} rows total)")


## Phase 7 — Evaluate P / R / F1 (macro + per-query, no-cap + K-cap)

In [ ]:
# =============================================================================
# PHASE 7 — Per-query and macro P/R/F1
# =============================================================================

with open(GOLD_JSON) as f:
    gold_sets = json.load(f)
print(f"Loaded gold sets from {GOLD_JSON.name}  ({len(gold_sets)} queries)")

def safe_f1(p, r):
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0

# --- No-cap eval (use all picks) ------------------------------------------
per_q = []
for qid in sorted(set(df.qid)):
    gold_set = set(gold_sets.get(qid, []))
    if not gold_set:
        print(f"  WARN: {qid} has no gold in {GOLD_JSON.name}")
        continue
    picks_set = set(all_picks[all_picks.qid == qid].did)
    correct = len(picks_set & gold_set)
    P = correct / max(1, len(picks_set))
    R = correct / max(1, len(gold_set))
    per_q.append({
        "qid": qid, "gold": len(gold_set), "picks": len(picks_set),
        "correct": correct, "P": P, "R": R, "F1": safe_f1(P, R),
    })

per_q_df = pd.DataFrame(per_q)
print(f"\nPer-query (no-cap):")
print(per_q_df.to_string(index=False))

macro_P  = per_q_df.P.mean()
macro_R  = per_q_df.R.mean()
macro_F1 = per_q_df.F1.mean()
print(f"\nMacro (no-cap):  P={macro_P:.4f}  R={macro_R:.4f}  F1={macro_F1:.4f}")

# --- K-cap eval (limit picks to gold count per query) ----------------------
# AUTO picks come first (most confident structural matches), then LLM by confidence desc.
def kcap_picks(qid, K):
    qpicks = all_picks[all_picks.qid == qid].copy()
    qpicks["_sortkey"] = qpicks.apply(
        lambda r: (0 if r.tier == "AUTO" else 1, -float(r.confidence)),
        axis=1,
    )
    qpicks = qpicks.sort_values("_sortkey").head(K)
    return set(qpicks.did)

per_q_kcap = []
for qid in sorted(set(df.qid)):
    gold_set = set(gold_sets.get(qid, []))
    if not gold_set: continue
    K = len(gold_set)
    picks_set = kcap_picks(qid, K)
    correct = len(picks_set & gold_set)
    P = correct / max(1, len(picks_set))
    R = correct / max(1, len(gold_set))
    per_q_kcap.append({
        "qid": qid, "K": K, "picks": len(picks_set),
        "correct": correct, "P": P, "R": R, "F1": safe_f1(P, R),
    })

per_q_kcap_df = pd.DataFrame(per_q_kcap)
print(f"\nPer-query (K-cap):")
print(per_q_kcap_df.to_string(index=False))

macro_P_kc  = per_q_kcap_df.P.mean()
macro_R_kc  = per_q_kcap_df.R.mean()
macro_F1_kc = per_q_kcap_df.F1.mean()
print(f"\nMacro (K-cap):   P={macro_P_kc:.4f}  R={macro_R_kc:.4f}  F1={macro_F1_kc:.4f}")

# --- Compare against v3 baseline -------------------------------------------
V3_F1 = 0.0822  # best from stage_b_grounded_llm_v3_outputs (K=500)
print(f"\n=== Comparison vs v3 (K=500) ===")
print(f"  v3 best Macro F1 (K-cap):  {V3_F1:.4f}")
print(f"  v5 Macro F1 (K-cap):       {macro_F1_kc:.4f}")
delta = macro_F1_kc - V3_F1
rel = (delta / V3_F1) * 100 if V3_F1 > 0 else 0.0
print(f"  delta:                      {delta:+.4f}  ({rel:+.1f}%)")
if macro_F1_kc >= 0.30:
    print(f"\n  CONCLUSIVE: F1 >= 0.30 - tiering on K=10k pool works, trajectory exists")
elif macro_F1_kc >= V3_F1 * 1.5:
    print(f"\n  PROMISING: 1.5x lift over v3 - real signal, needs more")
elif macro_F1_kc >= V3_F1:
    print(f"\n  MARGINAL: lift but small. Bottleneck remains LLM precision.")
else:
    print(f"\n  REGRESSION: bigger pool hurt. The K=500 + tiering was already near peak.")


## Phase 8 — Save metrics + summary

In [ ]:
# =============================================================================
# PHASE 8 — Save final metrics JSON
# =============================================================================

metrics = {
    "config": {
        "LLM_MODEL":           LLM_MODEL,
        "MAX_MODEL_LEN":       MAX_MODEL_LEN,
        "MAX_TOKENS":          MAX_TOKENS,
        "TEMPERATURE":         TEMPERATURE,
        "MAX_TEXT_CHARS_IN_PROMPT": MAX_TEXT_CHARS_IN_PROMPT,
        "MAX_QUESTION_CHARS":  MAX_QUESTION_CHARS,
        "stage_b_input":       str(STAGE_B_INPUT),
        "rows_total":          int(len(df)),
        "AUTO_COCIT_THRESH":      AUTO_COCIT_THRESH,
        "EVIDENCE_COCIT_THRESH":  EVIDENCE_COCIT_THRESH,
        "EVIDENCE_COSINE_THRESH": EVIDENCE_COSINE_THRESH,
    },
    "tier_totals": gtots.to_dict(orient="records"),
    "macro_no_cap": {"P": macro_P, "R": macro_R, "F1": macro_F1},
    "macro_k_cap":  {"P": macro_P_kc, "R": macro_R_kc, "F1": macro_F1_kc},
    "per_query_no_cap": {r["qid"]: r for r in per_q_df.to_dict(orient="records")},
    "per_query_k_cap":  {r["qid"]: r for r in per_q_kcap_df.to_dict(orient="records")},
    "diagnostics": {
        "parse_ok_rate":      parse_ok / max(1, n_llm),
        "keep_llm_rate":      n_keep / max(1, n_llm),
        "quote_in_text_rate": n_qok / max(1, n_keep),
        "keep_final_rate":    n_final / max(1, n_llm),
        "auto_tier_count":    int((df.tier == "AUTO").sum()),
        "llm_tier_count":     int((df.tier == "LLM").sum()),
        "drop_tier_count":    int((df.tier == "DROP").sum()),
        "auto_gold":          int(df[df.tier == "AUTO"].is_gold.sum()),
        "llm_tier_gold":      int(df[df.tier == "LLM"].is_gold.sum()),
        "drop_tier_gold":     int(df[df.tier == "DROP"].is_gold.sum()),
    },
}
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Metrics -> {METRICS_JSON.name}")

print(f"\n" + "="*78)
print(f"  STAGE B v5 (K=10000 + v3 tiering) — SUMMARY")
print(f"="*78)
print(f"  Input rows:           {len(df):,}")
print(f"  Tier AUTO / LLM / DROP:  "
      f"{(df.tier=='AUTO').sum():,} / "
      f"{(df.tier=='LLM').sum():,} / "
      f"{(df.tier=='DROP').sum():,}")
print(f"  Final picks:          {len(all_picks):,}")
print(f"  Macro F1 (no-cap):    {macro_F1:.4f}")
print(f"  Macro F1 (K-cap):     {macro_F1_kc:.4f}   <-- compare to v3=0.0822")
print(f"\nAll outputs at: {OUT_DIR}")
